In [ ]:
# ============================================================
# MEMBER B - CSV DIAGNOSTIC SCRIPT
# Run this first before writing BLIP captioning code
# ============================================================

import pandas as pd
import os

# ============================================================
# PATHS
# ============================================================
CSV_PATH = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs/blip_input_annotations.csv"
CROP_ROOT = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs/cropped_products"

# ============================================================
# 1. LOAD CSV
# ============================================================
df = pd.read_csv(CSV_PATH)

print("=" * 60)
print("1. BASIC INFO")
print("=" * 60)
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()

# ============================================================
# 2. SAMPLE ROWS
# ============================================================
print("=" * 60)
print("2. FIRST 3 ROWS (raw)")
print("=" * 60)
pd.set_option('display.max_colwidth', 120)
print(df.head(3).to_string())
print()

# ============================================================
# 3. NULL / EMPTY COUNTS
# ============================================================
print("=" * 60)
print("3. EMPTY FIELD COUNTS")
print("=" * 60)
for col in df.columns:
    empty = df[col].isna().sum() + (df[col] == "").sum()
    print(f"  {col}: {empty} empty / {len(df)} total")
print()

# ============================================================
# 4. HOW MANY IMAGES HAVE EACH CROP TYPE
# ============================================================
print("=" * 60)
print("4. CROP AVAILABILITY")
print("=" * 60)
has_upper = df['upper_body_crop'].notna() & (df['upper_body_crop'] != "")
has_lower = df['lower_body_crop'].notna() & (df['lower_body_crop'] != "")
has_full  = df['full_body_crop'].notna()  & (df['full_body_crop']  != "")

print(f"  Has upper_body crop : {has_upper.sum()}")
print(f"  Has lower_body crop : {has_lower.sum()}")
print(f"  Has full_body crop  : {has_full.sum()}")
print(f"  Has ALL 3 crops     : {(has_upper & has_lower & has_full).sum()}")
print(f"  Has NONE            : {(~has_upper & ~has_lower & ~has_full).sum()}")
print()

# ============================================================
# 5. SAMPLE CROP PATHS - check what they actually look like
# ============================================================
print("=" * 60)
print("5. SAMPLE CROP PATH VALUES")
print("=" * 60)
for col in ['upper_body_crop', 'lower_body_crop', 'full_body_crop']:
    sample = df[col].dropna().replace("", float('nan')).dropna().iloc[0] if df[col].notna().any() else "N/A"
    print(f"  {col}:\n    {sample}\n")

# ============================================================
# 6. DO CROP PATHS ACTUALLY EXIST ON DISK?
# ============================================================
print("=" * 60)
print("6. DISK EXISTENCE CHECK (first 5 non-empty per class)")
print("=" * 60)

for col in ['upper_body_crop', 'lower_body_crop', 'full_body_crop']:
    samples = df[col].dropna().replace("", float('nan')).dropna().head(5).tolist()
    print(f"\n  [{col}]")
    for p in samples:
        exists = os.path.exists(p)
        print(f"    {'OK' if exists else 'MISSING'} | {p}")

# ============================================================
# 7. WHAT DOES ACTUAL DISK FOLDER LOOK LIKE?
# ============================================================
print()
print("=" * 60)
print("7. ACTUAL FILES ON DISK (first 5 per folder)")
print("=" * 60)
for subfolder in ['upper_body', 'lower_body', 'full_body']:
    folder = os.path.join(CROP_ROOT, subfolder)
    if os.path.exists(folder):
        files = os.listdir(folder)[:5]
        print(f"\n  {subfolder}/ — total files: {len(os.listdir(folder))}")
        for f in files:
            print(f"    {f}")
    else:
        print(f"\n  {subfolder}/ — FOLDER NOT FOUND")

# ============================================================
# 8. PATH FORMAT MISMATCH CHECK
# ============================================================
print()
print("=" * 60)
print("8. PATH FORMAT — kaggle vs local check")
print("=" * 60)
sample_upper = df['upper_body_crop'].dropna().replace("", float('nan')).dropna().iloc[0]
print(f"  CSV stores paths like  : {sample_upper}")
print(f"  Starts with /kaggle/   : {str(sample_upper).startswith('/kaggle/')}")
print(f"  Starts with /memberA_  : {str(sample_upper).startswith('/memberA_')}")

# ============================================================
# 9. CONFIDENCE STATS
# ============================================================
print()
print("=" * 60)
print("9. CONFIDENCE SCORE STATS")
print("=" * 60)
for col in ['upper_body_confidence', 'lower_body_confidence', 'full_body_confidence']:
    vals = pd.to_numeric(df[col], errors='coerce').dropna()
    if len(vals) > 0:
        print(f"  {col}: min={vals.min():.3f}, max={vals.max():.3f}, mean={vals.mean():.3f}, count={len(vals)}")
    else:
        print(f"  {col}: no values found")

print()
print("=" * 60)
print("DIAGNOSTIC COMPLETE — share this output for BLIP planning")
print("=" * 60)